In [0]:
dbutils.widgets.text("in_catalog", "")
dbutils.widgets.text("in_schema", "")
dbutils.widgets.text("out_catalog", "")
dbutils.widgets.text("out_schema", "")

In [0]:
import sys
sys.path.append("../../src/")

In [0]:
from pyspark.sql.functions import col
from olist_lakehouse.readers import read_managed_table
params = dbutils.widgets.getAll()

In [0]:
order_items = read_managed_table(
    catalog=params["in_catalog"],
    schema=params["in_schema"],
    table="tbl_olist_order_items_dataset",
)
sellers = read_managed_table(
    catalog=params["in_catalog"],
    schema=params["in_schema"],
    table="tbl_olist_sellers_dataset",
)

In [0]:
order_items_join = order_items.join(sellers, on="seller_id", how="left")

In [0]:
order_items_transformed = order_items_join.select(
    "order_id",
    col("order_item_id").cast("int"),
    "product_id",
    col("price").cast("double"),
    col("freight_value").cast("double"),
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix",
)

In [0]:
(
    order_items_transformed
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{out_catalog}.{out_schema}.order_items")
)